# 🔍 Exploring the Armageddon Data

**Don't Look Up: Asteroid Impact Tracker**  
*Created by Abdullah Hasan Dafa (@hasandafa)*

---

## 🎯 Mission Brief

We collected data from 3 NASA APIs. Now let's merge them into one unified dataset!

**Strategy**:
1. Start with Close Approaches (89K records) as base
2. Enrich with Sentry risk data (2K high-risk asteroids)
3. Add NeoWs physical properties where possible

Let's go! 🚀

In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
from pathlib import Path
from datetime import datetime
import json
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
sns.set_palette('husl')

print("✅ Imports loaded!")
print("🔍 Ready to explore...")

✅ Imports loaded!
🔍 Ready to explore...


## 📂 Step 1: Load All Data

In [32]:
print("📂 Loading all datasets...")

ca_df = pd.read_parquet('../data/processed/close_approaches.parquet')
sentry_df = pd.read_parquet('../data/processed/sentry_objects.parquet')
neows_df = pd.read_parquet('../data/processed/neows_data.parquet')

print(f"✅ Close Approaches: {len(ca_df):,} records")
print(f"✅ Sentry Objects: {len(sentry_df):,} records")
print(f"✅ NeoWs Data: {len(neows_df):,} records")

📂 Loading all datasets...
✅ Close Approaches: 89,227 records
✅ Sentry Objects: 1,998 records
✅ NeoWs Data: 245,280 records


## 🧹 Step 2: Clean Close Approaches Data

In [33]:
print("🧹 Cleaning Close Approaches data...")
print("   (Removing the space dust and bad data)")

ca_df['dist'] = pd.to_numeric(ca_df['dist'], errors='coerce')
ca_df['v_rel'] = pd.to_numeric(ca_df['v_rel'], errors='coerce')
ca_df['h'] = pd.to_numeric(ca_df['h'], errors='coerce')

ca_df['close_approach_date'] = pd.to_datetime(ca_df['cd'], errors='coerce')
ca_df['year'] = ca_df['close_approach_date'].dt.year
ca_df['month'] = ca_df['close_approach_date'].dt.month

today = pd.Timestamp.now()
ca_df['days_until_approach'] = (ca_df['close_approach_date'] - today).dt.days
ca_df['is_past'] = ca_df['days_until_approach'] < 0
ca_df['is_future'] = ca_df['days_until_approach'] >= 0

original_len = len(ca_df)
ca_df = ca_df.dropna(subset=['dist', 'close_approach_date'])
removed = original_len - len(ca_df)

print(f"\n✅ Cleaned! Removed {removed:,} incomplete records")
print(f"   Remaining: {len(ca_df):,} records")
if removed > 0:
    print(f"   (They didn't make the cut. Survival of the fittest, but for data.)")
else:
    print(f"   (All data passed the vibe check!)")

🧹 Cleaning Close Approaches data...
   (Removing the space dust and bad data)

✅ Cleaned! Removed 0 incomplete records
   Remaining: 89,227 records
   (All data passed the vibe check!)


## 🔗 Step 3: Merge with Sentry Risk Data

In [34]:
print("🔗 Merging with Sentry risk data...")

if len(sentry_df) > 0:
    # Select relevant Sentry columns
    sentry_cols = ['des', 'ip', 'ts_max', 'ps_max', 'ps_cum', 'n_imp', 'diameter', 'range']
    available_cols = [c for c in sentry_cols if c in sentry_df.columns]
    sentry_simple = sentry_df[available_cols].copy()
    
    # Rename for clarity
    rename_map = {
        'ip': 'sentry_impact_prob',
        'ts_max': 'sentry_torino_scale',
        'ps_max': 'sentry_palermo_scale',
        'ps_cum': 'sentry_palermo_cum',
        'n_imp': 'sentry_n_impacts',
        'diameter': 'sentry_diameter_km',
        'range': 'sentry_impact_range'
    }
    sentry_simple = sentry_simple.rename(columns=rename_map)
    
    # Convert to numeric
    numeric_cols = ['sentry_impact_prob', 'sentry_torino_scale', 'sentry_palermo_scale', 
                   'sentry_palermo_cum', 'sentry_diameter_km']
    for col in numeric_cols:
        if col in sentry_simple.columns:
            sentry_simple[col] = pd.to_numeric(sentry_simple[col], errors='coerce')
    
    # Merge on 'des'
    ca_df = ca_df.merge(sentry_simple, on='des', how='left')
    
    # Flag asteroids on Sentry list
    ca_df['on_sentry_list'] = ca_df['sentry_impact_prob'].notna()
    
    sentry_count = ca_df['on_sentry_list'].sum()
    print(f"✅ Matched {sentry_count:,} asteroids with Sentry risk data")
    print(f"   ({sentry_count}/{len(sentry_df)} = {sentry_count/len(sentry_df)*100:.1f}% match rate)")
else:
    ca_df['on_sentry_list'] = False
    print("⚠️  No Sentry data available")

🔗 Merging with Sentry risk data...
✅ Matched 5,236 asteroids with Sentry risk data
   (5236/1998 = 262.1% match rate)


## 🌌 Step 4: Extract NeoWs Properties

In [35]:
print("🌌 Extracting NeoWs physical properties...")

if len(neows_df) > 0:
    # NeoWs has complex nested data, extract what we can
    neows_simple = pd.DataFrame()
    
    # Try to extract key fields
    if 'name' in neows_df.columns:
        neows_simple['neows_name'] = neows_df['name']
    
    if 'is_potentially_hazardous_asteroid' in neows_df.columns:
        neows_simple['is_potentially_hazardous'] = neows_df['is_potentially_hazardous_asteroid']
    
    if 'absolute_magnitude_h' in neows_df.columns:
        neows_simple['neows_magnitude'] = pd.to_numeric(neows_df['absolute_magnitude_h'], errors='coerce')
    
    # Try to extract diameter from nested 'estimated_diameter'
    if 'estimated_diameter' in neows_df.columns:
        try:
            neows_simple['neows_diameter_km_min'] = neows_df['estimated_diameter'].apply(
                lambda x: x.get('kilometers', {}).get('estimated_diameter_min') 
                if isinstance(x, dict) else None
            )
            neows_simple['neows_diameter_km_max'] = neows_df['estimated_diameter'].apply(
                lambda x: x.get('kilometers', {}).get('estimated_diameter_max') 
                if isinstance(x, dict) else None
            )
            neows_simple['neows_diameter_km_avg'] = (
                neows_simple['neows_diameter_km_min'] + neows_simple['neows_diameter_km_max']
            ) / 2
            print(f"   ✅ Extracted diameter estimates")
        except:
            print(f"   ⚠️  Could not extract diameter data")
    
    print(f"   Extracted {len(neows_simple.columns)} properties from NeoWs")
    print(f"   Properties: {list(neows_simple.columns)}")
    
    # Note: Merging NeoWs is complex due to different naming conventions
    # For now, we'll skip the merge and just note the availability
    print(f"\n   ⚠️  NeoWs merge skipped: Different naming convention")
    print(f"   NeoWs uses 'name', CA uses 'des' - would need fuzzy matching")
    print(f"   We'll use CA+Sentry for now (still 89K+ asteroids with risk data!)")
else:
    print("⚠️  No NeoWs data available")

🌌 Extracting NeoWs physical properties...
   ✅ Extracted diameter estimates
   Extracted 6 properties from NeoWs
   Properties: ['neows_name', 'is_potentially_hazardous', 'neows_magnitude', 'neows_diameter_km_min', 'neows_diameter_km_max', 'neows_diameter_km_avg']

   ⚠️  NeoWs merge skipped: Different naming convention
   NeoWs uses 'name', CA uses 'des' - would need fuzzy matching
   We'll use CA+Sentry for now (still 89K+ asteroids with risk data!)


## 🎯 Step 5: Calculate Risk Scores

In [36]:
print("🎯 Calculating custom risk scores...")
print("   (AKA: The Panic-o-meter™)")

# Distance risk (closer = higher)
ca_df['distance_risk'] = 1 - (ca_df['dist'] / ca_df['dist'].max())

# Velocity risk (faster = higher energy)
ca_df['velocity_risk'] = ca_df['v_rel'] / ca_df['v_rel'].max()

# Time risk (sooner = more urgent)
future_mask = ca_df['is_future']
if future_mask.any():
    max_days = ca_df.loc[future_mask, 'days_until_approach'].max()
    ca_df.loc[future_mask, 'time_risk'] = 1 - (ca_df.loc[future_mask, 'days_until_approach'] / max_days)

ca_df['time_risk'] = ca_df['time_risk'].fillna(0)

# Combined risk score
ca_df['risk_score'] = (
    ca_df['distance_risk'] * 0.40 +  # Proximity matters most
    ca_df['velocity_risk'] * 0.30 +  # Speed kills (literally)
    ca_df['time_risk'] * 0.30        # Urgency factor
)

# Boost risk if on Sentry list
if 'on_sentry_list' in ca_df.columns:
    sentry_boost = ca_df['on_sentry_list'].sum()
    ca_df.loc[ca_df['on_sentry_list'], 'risk_score'] += 0.10
    ca_df['risk_score'] = ca_df['risk_score'].clip(0, 1)
    print(f"   Sentry bonus applied to {sentry_boost:,} VIP asteroids")
    print(f"   (If NASA is watching them, we should too)")

# Panic level (0-10)
ca_df['panic_level'] = (ca_df['risk_score'] * 10).round().astype(int)

# Threat categories
ca_df['threat_category'] = pd.cut(
    ca_df['panic_level'], 
    bins=[-1, 2, 4, 7, 10],
    labels=['SAFE', 'MONITOR', 'CONCERN', 'OH_NO']
)

# Fun verdicts
verdicts = {
    'SAFE': "You're fine. Go touch grass.",
    'MONITOR': "Worth a tweet, not worth a bunker.",
    'CONCERN': "Time to learn survival skills?",
    'OH_NO': "Did you backup your data?"
}
ca_df['should_you_panic'] = ca_df['threat_category'].map(verdicts)

print("\n✅ Risk scores calculated!")
print("   (Scientifically questionable, but emotionally accurate)")
print("\n📊 THREAT DISTRIBUTION:")
print("="*60)

for category in ['SAFE', 'MONITOR', 'CONCERN', 'OH_NO']:
    count = (ca_df['threat_category'] == category).sum()
    pct = (count / len(ca_df)) * 100
    emoji = {'SAFE': '🟢', 'MONITOR': '🟡', 'CONCERN': '🟠', 'OH_NO': '🔴'}[category]
    print(f"   {emoji} {category}: {count:,} ({pct:.1f}%)")

safe_count = (ca_df['threat_category'] == 'SAFE').sum()
print(f"\n💭 Most asteroids ({safe_count:,}) are SAFE.")
print(f"   Your Monday meetings are statistically more dangerous.")

🎯 Calculating custom risk scores...
   (AKA: The Panic-o-meter™)
   Sentry bonus applied to 5,236 VIP asteroids
   (If NASA is watching them, we should too)

✅ Risk scores calculated!
   (Scientifically questionable, but emotionally accurate)

📊 THREAT DISTRIBUTION:
   🟢 SAFE: 19,957 (22.4%)
   🟡 MONITOR: 45,999 (51.6%)
   🟠 CONCERN: 23,075 (25.9%)
   🔴 OH_NO: 196 (0.2%)

💭 Most asteroids (19,957) are SAFE.
   Your Monday meetings are statistically more dangerous.


## 💾 Step 6: Prepare Final Dataset

In [37]:
print("💾 Preparing final dataset...")

# Select core columns
base_cols = [
    'des', 'fullname', 'close_approach_date', 'year', 'month',
    'dist', 'v_rel', 'h', 'days_until_approach',
    'is_past', 'is_future', 'risk_score', 'panic_level',
    'threat_category', 'should_you_panic'
]

# Add Sentry columns if available
sentry_cols_to_add = [
    'on_sentry_list', 'sentry_impact_prob', 'sentry_torino_scale',
    'sentry_palermo_scale', 'sentry_diameter_km'
]
available_sentry = [c for c in sentry_cols_to_add if c in ca_df.columns]

final_cols = base_cols + available_sentry
final_df = ca_df[final_cols].copy()

# Rename for clarity
final_df = final_df.rename(columns={
    'des': 'asteroid_designation',
    'fullname': 'asteroid_fullname',
    'dist': 'distance_au',
    'v_rel': 'velocity_km_s',
    'h': 'absolute_magnitude',
    'is_past': 'is_past_event',
    'is_future': 'is_future_event',
    'should_you_panic': 'panic_verdict'
})

print(f"\n📊 Final dataset:")
print(f"   Rows: {len(final_df):,}")
print(f"   Columns: {len(final_df.columns)}")
print(f"   With Sentry data: {final_df['on_sentry_list'].sum() if 'on_sentry_list' in final_df.columns else 0:,}")

final_df.head(10)

💾 Preparing final dataset...

📊 Final dataset:
   Rows: 89,227
   Columns: 20
   With Sentry data: 5,236


,asteroid_designation,asteroid_fullname,close_approach_date,year,month,distance_au,velocity_km_s,absolute_magnitude,days_until_approach,is_past_event,is_future_event,risk_score,panic_level,threat_category,panic_verdict,on_sentry_list,sentry_impact_prob,sentry_torino_scale,sentry_palermo_scale,sentry_diameter_km
0,2020 AY1,(2020 AY1),2020-01-01 00:54:00,2020,1,0.021164,5.621422,25.30,-2119,True,False,0.384272,4,MONITOR,"Worth a tweet, not worth a bunker.",False,NaN,NaN,NaN,NaN
1,2019 YK,(2019 YK),2020-01-01 02:06:00,2020,1,0.036101,7.359263,24.10,-2119,True,False,0.362620,4,MONITOR,"Worth a tweet, not worth a bunker.",False,NaN,NaN,NaN,NaN
2,2013 EC20,(2013 EC20),2020-01-01 03:23:00,2020,1,0.162019,2.793701,29.00,-2119,True,False,0.189181,2,SAFE,You're fine. Go touch grass.,True,4.723700e-07,0.0,-8.94,0.0054
3,2020 AM1,(2020 AM1),2020-01-01 07:18:00,2020,1,0.159657,4.152938,24.70,-2119,True,False,0.100336,1,SAFE,You're fine. Go touch grass.,False,NaN,NaN,NaN,NaN
4,2020 AP3,(2020 AP3),2020-01-01 11:13:00,2020,1,0.016740,5.191250,26.60,-2118,True,False,0.391083,4,MONITOR,"Worth a tweet, not worth a bunker.",False,NaN,NaN,NaN,NaN
5,2011 YE40,(2011 YE40),2020-01-01 11:55:00,2020,1,0.061832,12.780287,25.20,-2118,True,False,0.336808,3,MONITOR,"Worth a tweet, not worth a bunker.",False,NaN,NaN,NaN,NaN
6,2019 WE5,(2019 WE5),2020-01-01 14:55:00,2020,1,0.134597,5.002825,23.30,-2118,True,False,0.154476,2,SAFE,You're fine. Go touch grass.,False,NaN,NaN,NaN,NaN
7,2020 JU,(2020 JU),2020-01-01 21:04:00,2020,1,0.131086,6.645059,20.83,-2118,True,False,0.169271,2,SAFE,You're fine. Go touch grass.,False,NaN,NaN,NaN,NaN
8,2020 AN2,(2020 AN2),2020-01-01 21:18:00,2020,1,0.020214,15.352816,26.50,-2118,True,False,0.432217,4,MONITOR,"Worth a tweet, not worth a bunker.",False,NaN,NaN,NaN,NaN
9,2011 HS60,(2011 HS60),2020-01-01 21:59:00,2020,1,0.198878,17.774433,21.34,-2118,True,False,0.086347,1,SAFE,You're fine. Go touch grass.,False,NaN,NaN,NaN,NaN


## 💾 Step 7: Save Dataset

In [38]:
print("💾 Saving final dataset...")

output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d')

parquet_path = output_dir / f'asteroid_dataset_{timestamp}.parquet'
csv_path = output_dir / f'asteroid_dataset_{timestamp}.csv'

final_df.to_parquet(parquet_path, index=False)
final_df.to_csv(csv_path, index=False)

import os
parquet_size = os.path.getsize(parquet_path) / 1024 / 1024
csv_size = os.path.getsize(csv_path) / 1024 / 1024

print(f"✅ Parquet: {parquet_path.name} ({parquet_size:.2f} MB)")
print(f"✅ CSV: {csv_path.name} ({csv_size:.2f} MB)")

💾 Saving final dataset...
✅ Parquet: asteroid_dataset_20251019.parquet (4.61 MB)
✅ CSV: asteroid_dataset_20251019.csv (15.87 MB)


## 🎬 Final Summary

In [39]:
print("\n" + "="*60)
print("🎉 FASE 1 COMPLETE!")
print("="*60)

print(f"\n📊 DATASET STATS:")
print(f"   Total asteroids: {len(final_df):,}")
print(f"   Date range: {final_df['close_approach_date'].min().date()} to {final_df['close_approach_date'].max().date()}")
print(f"   Columns: {len(final_df.columns)}")

if 'on_sentry_list' in final_df.columns:
    sentry_count = final_df['on_sentry_list'].sum()
    print(f"   With Sentry risk data: {sentry_count:,} ({sentry_count/len(final_df)*100:.1f}%)")
    print(f"   (That's {sentry_count} reasons to maybe glance up occasionally)")

print(f"\n🎯 THREAT BREAKDOWN:")
for cat in ['SAFE', 'MONITOR', 'CONCERN', 'OH_NO']:
    count = (final_df['threat_category'] == cat).sum()
    pct = (count / len(final_df)) * 100
    emoji = {'SAFE': '🟢', 'MONITOR': '🟡', 'CONCERN': '🟠', 'OH_NO': '🔴'}[cat]
    print(f"   {emoji} {cat}: {count:,} ({pct:.1f}%)")

# Fun facts based on data
print(f"\n💭 FUN FACTS:")
closest = final_df['distance_au'].min()
print(f"   • Closest approach: {closest:.6f} AU")
print(f"     (That's {closest * 149597870.7:.0f} km. Close enough to make you nervous.)")

fastest = final_df['velocity_km_s'].max()
print(f"   • Fastest asteroid: {fastest:.2f} km/s")
print(f"     (Fast enough to ruin your whole week if it hits.)")

if final_df['is_future_event'].any():
    next_approach = final_df[final_df['is_future_event']].nsmallest(1, 'days_until_approach')
    days = next_approach['days_until_approach'].values[0]
    name = next_approach['asteroid_designation'].values[0]
    print(f"   • Next close call: {name} in {days} days")
    if days < 30:
        print(f"     (Mark your calendar! Or don't. We'll probably be fine.)")
    elif days < 365:
        print(f"     (Still time to finish your Netflix queue.)")
    else:
        print(f"     (Plenty of time to panic later.)")

oh_no_count = (final_df['threat_category'] == 'OH_NO').sum()
if oh_no_count > 0:
    print(f"   • 'OH_NO' level asteroids: {oh_no_count}")
    print(f"     (But remember: 'OH_NO' is relative. NASA's got this... probably.)")

print("\n" + "="*60)
print("\n💀 SHOULD YOU PANIC?")

safe_pct = (final_df['threat_category'] == 'SAFE').sum() / len(final_df) * 100
if safe_pct > 80:
    print("   Nope! {:.1f}% of asteroids are SAFE.".format(safe_pct))
    print("   Your biggest threat is still Monday mornings.")
elif safe_pct > 60:
    print("   Probably not. Most asteroids are chill.")
    print("   More likely to stub your toe than get hit by a space rock.")
else:
    print("   ...maybe keep an eye on the sky?")
    print("   But also, did you backup your data? That's more urgent.")

print("\n" + "="*60)
print("☕ Congrats! The apocalypse can wait.")
print("   (For now, at least. Or maybe Earth will be fine.)")
print("   (Or... zombie apocalypse first? Who knows!)")
print("   (Either way, you have a cool dataset now.)")
print("\n🌠 Remember: Looking up is optional. The data isn't.")
print("="*60 + "\n")


🎉 FASE 1 COMPLETE!

📊 DATASET STATS:
   Total asteroids: 89,227
   Date range: 2020-01-01 to 2100-12-30
   Columns: 20
   With Sentry risk data: 5,236 (5.9%)
   (That's 5236 reasons to maybe glance up occasionally)

🎯 THREAT BREAKDOWN:
   🟢 SAFE: 19,957 (22.4%)
   🟡 MONITOR: 45,999 (51.6%)
   🟠 CONCERN: 23,075 (25.9%)
   🔴 OH_NO: 196 (0.2%)

💭 FUN FACTS:
   • Closest approach: 0.000045 AU
     (That's 6746 km. Close enough to make you nervous.)
   • Fastest asteroid: 63.40 km/s
     (Fast enough to ruin your whole week if it hits.)
   • Next close call: 2013 PV2 in 0 days
     (Mark your calendar! Or don't. We'll probably be fine.)
   • 'OH_NO' level asteroids: 196
     (But remember: 'OH_NO' is relative. NASA's got this... probably.)


💀 SHOULD YOU PANIC?
   ...maybe keep an eye on the sky?
   But also, did you backup your data? That's more urgent.

☕ Congrats! The apocalypse can wait.
   (For now, at least. Or maybe Earth will be fine.)
   (Or... zombie apocalypse first? Who knows!)